# Apache Iceberg with Spark and Polaris REST Catalog

This notebook demonstrates Apache Iceberg with Spark using the **Polaris REST Catalog** backed by **MinIO** (S3-compatible storage).

In [ ]:
from pyspark.sql import SparkSession

print("🔐 Polaris: http://polaris:8181")
print("💾 MinIO: http://minio:9000")
print()
print("ℹ️  All Spark/Iceberg configuration loaded from spark-defaults.conf")
print("   (packages, extensions, catalogs, S3 settings)")
print()

# Create Spark session - configuration is already set via spark-defaults.conf
spark = SparkSession.builder \
    .appName("IcebergPolaris") \
    .getOrCreate()

print(f"✅ Spark {spark.version} ready!")
print(f"📁 Default Catalog: {spark.conf.get('spark.sql.defaultCatalog', 'spark_catalog')}")


## Show Catalogs & Namespaces

In [ ]:
spark.sql("SHOW CATALOGS").show()
spark.sql("SHOW NAMESPACES IN polaris").show()

## Create Namespace and Table

In [ ]:
# Create namespace (schema) if it doesn't exist
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.demo")
print("✅ Namespace created!")

# Create table
spark.sql("""
    CREATE TABLE IF NOT EXISTS polaris.demo.employees (
        id INT,
        name STRING,
        age INT,
        department STRING
    ) USING iceberg
""")
print("✅ Table created!")

## Insert Data

In [ ]:
spark.sql("""
    INSERT INTO polaris.demo.employees VALUES
    (1, 'Alice', 28, 'Engineering'),
    (2, 'Bob', 35, 'Sales'),
    (3, 'Charlie', 42, 'Marketing')
""")
print("✅ Data inserted!")

## Query Table

In [ ]:
spark.table("polaris.demo.employees").show()

In [ ]:
spark.sql("""
    SELECT department, COUNT(*) as cnt, AVG(age) as avg_age
    FROM polaris.demo.employees
    GROUP BY department
""").show()

## ACID - Append Data

In [ ]:
spark.sql("""
    INSERT INTO polaris.demo.employees VALUES
    (4, 'Diana', 31, 'Engineering'),
    (5, 'Eve', 29, 'Sales')
""")
print("✅ Data appended\n")
spark.table("polaris.demo.employees").show()

## Update Records

In [ ]:
spark.sql("""
    UPDATE polaris.demo.employees
    SET department = 'Product'
    WHERE name = 'Alice'
""")
print("✅ Record updated\n")
spark.table("polaris.demo.employees").show()

## Delete Records

In [ ]:
spark.sql("""
    DELETE FROM polaris.demo.employees
    WHERE name = 'Bob'
""")
print("✅ Record deleted\n")
spark.table("polaris.demo.employees").show()

## Snapshots & History

In [ ]:
print("📸 Snapshots:")
spark.sql("SELECT snapshot_id, operation FROM polaris.demo.employees.snapshots").show(truncate=False)

In [ ]:
print("📜 History:")
spark.sql("SELECT * FROM polaris.demo.employees.history").show(truncate=False)

## Time Travel

In [ ]:
snapshots = spark.sql("SELECT snapshot_id FROM polaris.demo.employees.snapshots ORDER BY committed_at").collect()
if len(snapshots) > 0:
    first_snapshot = snapshots[0].snapshot_id
    print(f"🕐 Reading from first snapshot: {first_snapshot}\n")
    spark.read.option("snapshot-id", first_snapshot).table("polaris.demo.employees").show()

## Schema Evolution

In [ ]:
spark.sql("ALTER TABLE polaris.demo.employees ADD COLUMN salary DOUBLE")
print("✅ Column added\n")
spark.table("polaris.demo.employees").printSchema()

## Metadata Tables

In [ ]:
print("📊 Files:")
spark.sql("SELECT file_path, record_count FROM polaris.demo.employees.files").show(truncate=False)